# 第11课 - 代理到代理 (A2A) 协议


## 设置


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## What is the A2A Protocol?

The **Agent-to-Agent (A2A) protocol** is an open standard that enables AI agents to communicate,
discover each other, and collaborate — even when they are built on different frameworks or hosted
by different services.

Key concepts:

- **Discovery** – Agents publish an *Agent Card* that describes their capabilities, making it
  easy for other agents (or orchestrators) to find the right specialist for a task.
- **Message Passing** – Agents exchange structured messages through a common protocol, so a
  request from one agent can be understood and fulfilled by another regardless of its internal
  implementation.
- **Task Lifecycle** – A2A defines states such as *submitted*, *working*, *completed*, and
  *failed*, giving the orchestrator full visibility into how a delegated task is progressing.

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents
into a workflow where each agent contributes its expertise and passes results to the next.


## 创建专业旅游代理商


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## 通过工作流程实现多智能体协作

我们将三个智能体连接成一个顺序工作流程，模拟 A2A 消息传递：

1. **CurrencyExchangeAgent** 接收用户请求并提供货币指导。
2. **ActivityPlannerAgent** 接收丰富的上下文并添加活动推荐。
3. **TravelManagerAgent** 综合两个输入，生成最终的旅行简报。


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 理解生产环境中的 A2A

在生产环境中，A2A 协议解锁了强大的跨服务场景：

| 能力 | 描述 |
|---|---|
| <strong>跨框架互操作</strong> | 使用一个框架构建的代理可以将任务委托给任何其他符合 A2A 标准的框架构建的代理，实现真正的跨组织互操作。 |
| <strong>服务边界</strong> | 代理可以分布在不同的微服务、云区域甚至不同的组织中，同时仍能无缝协作。 |
| <strong>动态发现</strong> | 编排器可以在运行时查询代理卡注册表，找到最适合特定子任务的专家。 |
| <strong>流式传输与推送通知</strong> | A2A 支持服务器发送事件（SSE）用于实时进度更新，以及长时间运行任务的推送通知。 |

我们上面构建的工作流是该模式的简化进程内版本。在实际
部署中，每个代理都会暴露 HTTP 端点，发布代理卡，并通过
A2A JSON-RPC 协议进行通信。


## Summary

In this lesson you learned:

1. **What the A2A protocol is** — an open standard for agent-to-agent discovery, messaging,
   and task management.
2. **How to create specialized agents** — a Currency Exchange agent, an Activity Planner agent,
   and a Travel Manager orchestrator.
3. **How to wire agents into a workflow** — using `WorkflowBuilder` to model sequential
   message passing between agents.
4. **How A2A works in production** — enabling cross-framework, cross-service collaboration
   with dynamic discovery and streaming updates.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
